# 04. Weather Feature Engineering
**목적**: 기상 원자료에서 화재위험 의미의 파생변수를 생성하고, 전력설비별·날짜별 기상 feature를 산출한다.

산출 feature 그룹:
1. Rolling window feature (1/3/7/14일)
2. 단위 기상 위험 점수 (DrynessScore, WindScore, HeatScore, NoRainScore)
3. 상호작용 feature (dry_wind, hot_dry, no_rain_wind)
4. 최종 Weather Hazard Score (0~100)

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from config import (DATA_RAW, DATA_PROCESSED, ROLL_WINDOWS, INTERACTION_PAIRS)

In [ ]:
with open(DATA_PROCESSED / 'column_map.json', encoding='utf-8') as f:
    cmap = json.load(f)

WC = cmap['weather']
FC = cmap['facility']

df_w = pd.read_csv(DATA_RAW / 'weather.csv', encoding='utf-8-sig')
df_w[WC['date']] = pd.to_datetime(df_w[WC['date']])

print(f'기상 데이터: {df_w.shape}')
df_w.head()

## 1. 컬럼 표준화

In [ ]:
# 표준 컬럼명으로 rename (column_map 기반)
rename_map = {
    WC['date']:        'date',
    WC['station_id']:  'stn_id',
    WC['temperature']: 'temp',
    WC['humidity']:    'rh',
    WC['wind_speed']:  'ws',
    WC['wind_dir']:    'wd',
    WC['precip']:      'precip',
}
if WC['eff_humidity']:
    rename_map[WC['eff_humidity']] = 'eff_rh'

rename_map = {k: v for k, v in rename_map.items() if k is not None}
df = df_w.rename(columns=rename_map)
df = df.sort_values(['stn_id', 'date']).reset_index(drop=True)
print(df.columns.tolist())

## 2. Rolling Window Features

In [ ]:
def make_rolling_features(df, group_col, date_col, windows):
    """관측소별 rolling window 통계를 생성한다."""
    df = df.sort_values([group_col, date_col]).copy()

    for w in windows:
        g = df.groupby(group_col)

        df[f'temp_max_{w}d']     = g['temp'].transform(lambda x: x.rolling(w, min_periods=1).max())
        df[f'temp_mean_{w}d']    = g['temp'].transform(lambda x: x.rolling(w, min_periods=1).mean())
        df[f'rh_min_{w}d']       = g['rh'].transform(lambda x: x.rolling(w, min_periods=1).min())
        df[f'rh_mean_{w}d']      = g['rh'].transform(lambda x: x.rolling(w, min_periods=1).mean())
        df[f'ws_max_{w}d']       = g['ws'].transform(lambda x: x.rolling(w, min_periods=1).max())
        df[f'ws_mean_{w}d']      = g['ws'].transform(lambda x: x.rolling(w, min_periods=1).mean())
        df[f'precip_sum_{w}d']   = g['precip'].transform(lambda x: x.rolling(w, min_periods=1).sum())

    # 무강수 연속일
    df['dry_day_flag'] = (df['precip'] < 0.1).astype(int)
    def no_rain_streak(s):
        streak = s.groupby((s != s.shift()).cumsum()).transform('cumsum')
        return streak * s
    df['no_rain_days'] = df.groupby(group_col)['dry_day_flag'].transform(no_rain_streak)

    return df


df = make_rolling_features(df, 'stn_id', 'date', ROLL_WINDOWS)
print(f'Rolling feature 생성 완료. 컬럼 수: {df.shape[1]}')

## 3. 단위 위험 점수 (0~100 정규화)

In [ ]:
def percentile_score(series, ascending=True):
    """
    값을 0~100 백분위 점수로 변환한다.
    ascending=True  → 높을수록 위험 (기온, 풍속 등)
    ascending=False → 낮을수록 위험 (습도, 강수량 등)
    """
    rank = series.rank(pct=True)
    return (rank * 100) if ascending else ((1 - rank) * 100)


# 3일 창 기준 단위 점수 (보고서에서 사용)
df['dryness_score'] = percentile_score(df['rh_min_3d'],     ascending=False)
df['wind_score']    = percentile_score(df['ws_max_1d'],     ascending=True)
df['heat_score']    = percentile_score(df['temp_max_1d'],   ascending=True)
df['no_rain_score'] = percentile_score(df['no_rain_days'],  ascending=True)

# 실효습도가 있으면 건조도 점수에 반영
if 'eff_rh' in df.columns:
    eff_score = percentile_score(df['eff_rh'], ascending=False)
    df['dryness_score'] = (df['dryness_score'] * 0.6 + eff_score * 0.4)

print('단위 위험 점수 생성 완료')
df[['dryness_score', 'wind_score', 'heat_score', 'no_rain_score']].describe()

## 4. 상호작용 Feature

In [ ]:
# dry_wind_interaction = DrynessScore × WindScore / 100
df['dry_wind_interaction']   = df['dryness_score'] * df['wind_score'] / 100
df['hot_dry_interaction']    = df['heat_score']    * df['dryness_score'] / 100
df['no_rain_wind_interaction'] = df['no_rain_score'] * df['wind_score'] / 100

print('상호작용 feature 생성 완료')
df[['dry_wind_interaction', 'hot_dry_interaction', 'no_rain_wind_interaction']].describe()

## 4-1. 기상특보 파생 Flag (기상청 발령 기준 직접 계산)

기상청 공식 특보 발령 기준을 그대로 적용 → 실제 특보 데이터 없이도 동일 정보 생성

| 특보 | 기준 |
|------|------|
| **건조주의보** | 실효습도 35% 이하 또는 최저습도 25% 이하 |
| **강풍주의보** | 평균풍속 14m/s 이상 또는 순간최대풍속 20m/s 이상 |
| **건조경보** | 실효습도 25% 이하 또는 최저습도 15% 이하 |
| **강풍경보** | 평균풍속 21m/s 이상 또는 순간최대풍속 26m/s 이상 |

In [ ]:
# 산불위험실황분석 체계(기상청)와 동일한 변수 조합
# 기상특보 flag를 interaction_score에 반영
df['interaction_score'] = (
    df['dry_wind_interaction']    * 0.40 +
    df['no_rain_wind_interaction'] * 0.25 +
    df['hot_dry_interaction']      * 0.15 +
    df['combined_risk_flag']       * 20   +   # 건조+강풍 동시: 직접 가산
    df['dry_warn_flag']            * 10        # 건조경보: 추가 가산
).clip(0, 100)

df['weather_hazard'] = (
    0.25 * df['dryness_score'] +
    0.25 * df['wind_score'] +
    0.20 * df['heat_score'] +
    0.15 * df['no_rain_score'] +
    0.15 * df['interaction_score']
).clip(0, 100)

print('Weather Hazard Score 생성 완료')
print(df['weather_hazard'].describe())
print(f"\n복합위험일 평균 hazard: {df[df['combined_risk_flag']==1]['weather_hazard'].mean():.1f}")
print(f"일반일    평균 hazard: {df[df['combined_risk_flag']==0]['weather_hazard'].mean():.1f}")

## 5. Weather Hazard Score 산출

In [ ]:
import geopandas as gpd
gdf_fac = gpd.read_file(DATA_PROCESSED / 'facility_proj.gpkg')
FID_COL = FC['facility_id']
df_match = gdf_fac[[FID_COL, 'nearest_stn_id']].copy()

# 기상 feature 컬럼 목록 (특보 flag 추가)
warning_cols = [
    'dry_watch_flag', 'dry_warn_flag',
    'wind_watch_flag', 'wind_warn_flag',
    'combined_risk_flag', 'combined_risk_days', 'dry_watch_days'
]

weather_feature_cols = (
    ['stn_id', 'date', 'weather_hazard',
     'dryness_score', 'wind_score', 'heat_score', 'no_rain_score',
     'dry_wind_interaction', 'hot_dry_interaction', 'no_rain_wind_interaction',
     'no_rain_days']
    + warning_cols
    + [f'temp_max_{w}d'   for w in ROLL_WINDOWS]
    + [f'rh_min_{w}d'     for w in ROLL_WINDOWS]
    + [f'ws_max_{w}d'     for w in ROLL_WINDOWS]
    + [f'precip_sum_{w}d' for w in ROLL_WINDOWS]
)
weather_feature_cols = [c for c in weather_feature_cols if c in df.columns]

df_weather_features = df[weather_feature_cols].copy()

facility_weather = df_match.merge(
    df_weather_features,
    left_on='nearest_stn_id',
    right_on='stn_id',
    how='left'
).drop(columns=['stn_id', 'nearest_stn_id'])

print(f'설비-날짜 기상 feature: {facility_weather.shape}')
print(f'feature 수: {facility_weather.shape[1] - 2} (FID·날짜 제외)')
facility_weather.head()

## 6. 설비별 기상 feature 매핑

In [ ]:
import pickle

# 설비-관측소 매칭 로드
import geopandas as gpd
gdf_fac = gpd.read_file(DATA_PROCESSED / 'facility_proj.gpkg')
FID_COL = FC['facility_id']

# 최근접 관측소 ID 컬럼 추출
df_match = gdf_fac[[FID_COL, 'nearest_stn_id']].copy()

# 기상 feature 컬럼 목록
weather_feature_cols = (
    ['stn_id', 'date', 'weather_hazard',
     'dryness_score', 'wind_score', 'heat_score', 'no_rain_score',
     'dry_wind_interaction', 'hot_dry_interaction', 'no_rain_wind_interaction',
     'no_rain_days']
    + [f'temp_max_{w}d' for w in ROLL_WINDOWS]
    + [f'rh_min_{w}d' for w in ROLL_WINDOWS]
    + [f'ws_max_{w}d' for w in ROLL_WINDOWS]
    + [f'precip_sum_{w}d' for w in ROLL_WINDOWS]
)
weather_feature_cols = [c for c in weather_feature_cols if c in df.columns]

df_weather_features = df[weather_feature_cols].copy()

# 설비-날짜 단위로 cross join (설비 × 날짜)
facility_weather = df_match.merge(
    df_weather_features,
    left_on='nearest_stn_id',
    right_on='stn_id',
    how='left'
).drop(columns=['stn_id', 'nearest_stn_id'])

print(f'설비-날짜 기상 feature: {facility_weather.shape}')
facility_weather.head()

In [ ]:
facility_weather.to_parquet(DATA_PROCESSED / 'weather_features.parquet', index=False)
print('저장 완료: weather_features.parquet')
print('\n다음 단계: 05_risk_index.ipynb')